# 03 – Đánh giá cuối & phân tích lỗi

**Nguyên tắc:** test set chỉ mở **một lần**, sau khi checkpoint đã được khóa bởi notebook 02.  
Notebook này:
1. Đánh giá tất cả model E0–E3 (seed 42) trên tập test → bảng so sánh
2. Classification report + confusion matrix cho model tốt nhất (E3)
3. Phân tích ảnh dự đoán sai (ít nhất 6 ví dụ)
4. Đẩy kết quả test lên W&B

## Cell 1 – Setup

In [ ]:
from pathlib import Path
import sys, os, json
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from PIL import Image

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from src.data import make_dataloaders, make_transforms
from src.model import SimpleCNN, build_resnet18
from src.train import run_epoch

try:
    import wandb; _WANDB_OK = True
except ImportError:
    wandb = None; _WANDB_OK = False

WANDB_PROJECT = os.getenv('WANDB_PROJECT', 'plant-disease-deep-learning')
WANDB_ENTITY  = os.getenv('WANDB_ENTITY', None)

DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
metadata = pd.read_csv(PROJECT_ROOT / 'data' / 'metadata.csv')
LABELS   = metadata.sort_values('label').drop_duplicates('label')['class_name'].tolist()
NUM_CLASSES = len(LABELS)
EVAL_SEED = 42   # seed dùng để load checkpoint

# Dataloader test (không augmentation)
loaders_eval = make_dataloaders(metadata, batch_size=32, augmentation=False)

print(f'Device={DEVICE}, classes={NUM_CLASSES}')

## Cell 2 – Hàm tiện ích: load checkpoint + đánh giá test

In [ ]:
def load_model(exp_name: str, seed: int, freeze: bool | None, is_cnn: bool):
    """Load checkpoint cho (experiment, seed). Trả về model eval-mode."""
    ckpt_path = PROJECT_ROOT / 'checkpoints' / f'{exp_name}_seed{seed}.pt'
    if not ckpt_path.exists():
        raise FileNotFoundError(f'Checkpoint không tồn tại: {ckpt_path}')
    if is_cnn:
        model = SimpleCNN(NUM_CLASSES)
    else:
        model = build_resnet18(NUM_CLASSES, freeze_backbone=(freeze if freeze else False))
    state = torch.load(ckpt_path, map_location=DEVICE, weights_only=True)
    model.load_state_dict(state['model_state'])
    model.to(DEVICE).eval()
    return model, state.get('val_macro_f1', None)


def evaluate_test(model) -> dict:
    """Trả về y_true, y_pred và metrics trên test set."""
    y_true, y_pred = [], []
    with torch.no_grad():
        for images, labels in loaders_eval['test']:
            preds = model(images.to(DEVICE)).argmax(1).cpu()
            y_true.extend(labels.tolist())
            y_pred.extend(preds.tolist())
    acc      = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    return {'y_true': y_true, 'y_pred': y_pred, 'accuracy': acc, 'macro_f1': macro_f1}

print('Hàm tiện ích đã sẵn sàng.')

## Cell 3 – Bảng so sánh test: E0 vs E1 vs E2 vs E3

In [ ]:
MODELS_INFO = [
    # (exp_name, freeze, is_cnn, label)
    ('E0_simplecnn',          None,  True,  'E0 SimpleCNN'),
    ('E1_resnet_frozen',      True,  False, 'E1 ResNet-18 Frozen'),
    ('E2_resnet_finetune',    False, False, 'E2 ResNet-18 Fine-tune'),
    ('E3_resnet_finetune_aug',False, False, 'E3 ResNet-18 Fine-tune + Aug'),
]

test_results = {}
comparison_rows = []

for exp_name, freeze, is_cnn, label in MODELS_INFO:
    try:
        model, best_val_f1 = load_model(exp_name, EVAL_SEED, freeze, is_cnn)
        res = evaluate_test(model)
        test_results[exp_name] = res
        comparison_rows.append({
            'Model'         : label,
            'Best Val F1'   : round(best_val_f1, 4) if best_val_f1 else 'N/A',
            'Test Accuracy' : round(res['accuracy'], 4),
            'Test Macro-F1' : round(res['macro_f1'], 4),
        })
        print(f'  {label}: accuracy={res["accuracy"]:.4f}  macro_f1={res["macro_f1"]:.4f}')
    except FileNotFoundError as e:
        print(f'  Bỏ qua {label}: {e}')
        comparison_rows.append({'Model': label, 'Best Val F1': 'missing',
                                 'Test Accuracy': 'missing', 'Test Macro-F1': 'missing'})

comp_df = pd.DataFrame(comparison_rows)
print('\n=== Bảng so sánh mô hình ===\n')
print(comp_df.to_string(index=False))
comp_df.to_csv(PROJECT_ROOT / 'outputs' / 'test_comparison.csv', index=False)

## Cell 4 – Ablation delta (trả lời RQ1 & RQ2)

In [ ]:
def get_f1(exp_name):
    if exp_name in test_results:
        return test_results[exp_name]['macro_f1']
    return None

f1_E1 = get_f1('E1_resnet_frozen')
f1_E2 = get_f1('E2_resnet_finetune')
f1_E3 = get_f1('E3_resnet_finetune_aug')

print('=== Ablation Studies ===\n')
if f1_E1 and f1_E2:
    delta_rq1 = f1_E2 - f1_E1
    verdict = 'Ủng hộ H1 ✓' if delta_rq1 > 0 else 'Không ủng hộ H1 ✗'
    print(f'RQ1 – Fine-tune vs Frozen: ΔF1 = {delta_rq1:+.4f}  → {verdict}')

if f1_E2 and f1_E3:
    delta_rq2 = f1_E3 - f1_E2
    verdict = 'Ủng hộ H2 ✓' if delta_rq2 > 0 else 'Không ủng hộ H2 ✗'
    print(f'RQ2 – Aug vs No-aug:        ΔF1 = {delta_rq2:+.4f}  → {verdict}')

ablation_df = pd.DataFrame([
    {'Câu hỏi': 'RQ1', 'So sánh': 'E2 − E1',
     'ΔMacro-F1 (test)': round(f1_E2 - f1_E1, 4) if f1_E1 and f1_E2 else 'N/A'},
    {'Câu hỏi': 'RQ2', 'So sánh': 'E3 − E2',
     'ΔMacro-F1 (test)': round(f1_E3 - f1_E2, 4) if f1_E2 and f1_E3 else 'N/A'},
])
print('\n'); print(ablation_df.to_string(index=False))
ablation_df.to_csv(PROJECT_ROOT / 'outputs' / 'ablation_results.csv', index=False)

## Cell 5 – Confusion matrix (model tốt nhất = E3)

In [ ]:
BEST_EXP = 'E3_resnet_finetune_aug'
if BEST_EXP not in test_results:
    print(f'Chưa có kết quả cho {BEST_EXP}. Chạy Cell 3 trước.')
else:
    y_true = test_results[BEST_EXP]['y_true']
    y_pred = test_results[BEST_EXP]['y_pred']

    matrix = confusion_matrix(y_true, y_pred, normalize='true')
    fig, ax = plt.subplots(figsize=(16, 14))
    # Chỉ hiện annotation nếu số lớp nhỏ
    annot = NUM_CLASSES <= 20
    sns.heatmap(matrix, cmap='Blues', xticklabels=LABELS, yticklabels=LABELS,
                annot=annot, fmt='.2f' if annot else '', ax=ax)
    ax.set_xlabel('Dự đoán', fontsize=11)
    ax.set_ylabel('Nhãn thật', fontsize=11)
    ax.set_title(f'Confusion Matrix – {BEST_EXP} (normalized, seed={EVAL_SEED})', fontsize=12)
    plt.xticks(rotation=90, fontsize=7)
    plt.yticks(rotation=0,  fontsize=7)
    plt.tight_layout()
    out_path = PROJECT_ROOT / 'outputs' / 'confusion_matrix_test.png'
    plt.savefig(out_path, dpi=160)
    plt.show()
    print(f'Đã lưu: {out_path}')

## Cell 6 – Classification report (per-class F1)

In [ ]:
if BEST_EXP in test_results:
    y_true = test_results[BEST_EXP]['y_true']
    y_pred = test_results[BEST_EXP]['y_pred']

    report_str = classification_report(y_true, y_pred,
                                        target_names=LABELS, zero_division=0)
    print(report_str)

    # Lưu dạng dict để đưa vào báo cáo
    report_dict = classification_report(
        y_true, y_pred, target_names=LABELS, zero_division=0, output_dict=True
    )
    per_class_df = pd.DataFrame(report_dict).T.round(4)
    per_class_df.to_csv(PROJECT_ROOT / 'outputs' / 'per_class_report.csv')

    # Top 10 lớp có F1 thấp nhất
    class_rows = per_class_df.loc[[l for l in LABELS if l in per_class_df.index]]
    print('\n=== 10 lớp có F1 thấp nhất ===\n')
    print(class_rows['f1-score'].sort_values().head(10).to_string())

## Cell 7 – Phân tích ảnh dự đoán sai (≥6 ví dụ)

In [ ]:
if BEST_EXP in test_results:
    y_true = np.array(test_results[BEST_EXP]['y_true'])
    y_pred = np.array(test_results[BEST_EXP]['y_pred'])
    wrong_idx = np.where(y_true != y_pred)[0]
    print(f'Tổng sai: {len(wrong_idx)} / {len(y_true)}')

    # Load model để lấy xác suất
    model_e3, _ = load_model(BEST_EXP, EVAL_SEED, False, False)

    test_meta = metadata[metadata['split'] == 'test'].reset_index(drop=True)

    _, eval_tf = make_transforms(augmentation=False)

    # Chọn 12 ảnh sai ngẫu nhiên (seed 0 để tái lập)
    rng = np.random.default_rng(0)
    sample_idx = rng.choice(wrong_idx, size=min(12, len(wrong_idx)), replace=False)

    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    axes = axes.flatten()

    for plot_i, si in enumerate(sample_idx):
        row = test_meta.iloc[si]
        img = Image.open(row['path']).convert('RGB')
        with torch.no_grad():
            probs = model_e3(eval_tf(img).unsqueeze(0).to(DEVICE)).softmax(1)[0].cpu()
        pred_label   = LABELS[int(y_pred[si])]
        true_label   = LABELS[int(y_true[si])]
        pred_conf    = float(probs[int(y_pred[si])])

        axes[plot_i].imshow(img)
        axes[plot_i].axis('off')
        axes[plot_i].set_title(
            f'True: {true_label.split("___")[-1]}\n'
            f'Pred: {pred_label.split("___")[-1]} ({pred_conf:.2%})',
            fontsize=7, color='red'
        )

    for ax in axes[len(sample_idx):]:
        ax.set_visible(False)

    plt.suptitle(f'Ví dụ dự đoán sai – {BEST_EXP} (seed={EVAL_SEED})', fontsize=12)
    plt.tight_layout()
    out_path = PROJECT_ROOT / 'outputs' / 'error_examples.png'
    plt.savefig(out_path, dpi=160)
    plt.show()
    print(f'Đã lưu: {out_path}')

## Cell 8 – Upload kết quả test lên W&B (1 run tổng hợp)

In [ ]:
if _WANDB_OK and test_results:
    wb_run = wandb.init(
        project  = WANDB_PROJECT,
        entity   = WANDB_ENTITY,
        name     = f'evaluation_seed{EVAL_SEED}',
        job_type = 'evaluation',
        tags     = ['evaluation', f'seed-{EVAL_SEED}'],
    )

    # Log kết quả test từng model
    for exp_name, _, _, label in MODELS_INFO:
        if exp_name in test_results:
            r = test_results[exp_name]
            wb_run.summary[f'{exp_name}/test_accuracy'] = r['accuracy']
            wb_run.summary[f'{exp_name}/test_macro_f1'] = r['macro_f1']

    # Upload ảnh
    for img_name in ['confusion_matrix_test.png', 'error_examples.png',
                     'validation_f1_comparison.png', 'train_val_curves_seed42.png']:
        img_path = PROJECT_ROOT / 'outputs' / img_name
        if img_path.exists():
            wb_run.log({img_name.replace('.png',''): wandb.Image(str(img_path))})

    # Upload CSV
    for csv_name in ['test_comparison.csv', 'ablation_results.csv', 'per_class_report.csv']:
        csv_path = PROJECT_ROOT / 'outputs' / csv_name
        if csv_path.exists():
            artifact = wandb.Artifact(csv_name.replace('.csv',''), type='result')
            artifact.add_file(str(csv_path))
            wb_run.log_artifact(artifact)

    wb_run.finish()
    print('Upload W&B hoàn tất.')
else:
    print('Bỏ qua W&B upload (wandb không khả dụng hoặc chưa có kết quả).')